# Fauna V6: oclusión en 582 casos aceptados

Este complemento verifica la revisión humana consolidada y presenta métricas ya calculadas sobre las predicciones congeladas de V6. No entrena ni ejecuta inferencia.

**Criterios:** conservar exactamente los 582 casos aceptados, excluir los 18 sin nivel, usar la revisión de Miguel en el caso 121, no completar notas ausentes y comprobar los intervalos de 200 réplicas por nivel. Ejecutar desde esta carpeta, junto con `data/`.

In [1]:
from pathlib import Path
import csv
import hashlib
import json
from collections import Counter

data = Path.cwd() / "data"
integrity = json.loads((data / "integrity.json").read_text(encoding="utf-8"))
for name, expected in integrity["sha256"].items():
    assert hashlib.sha256((data / name).read_bytes()).hexdigest() == expected, name
results = json.loads((data / "results.json").read_text(encoding="utf-8"))
acceptance = json.loads((data / "review_acceptance.json").read_text(encoding="utf-8"))
protocol = json.loads((data / "protocol.json").read_text(encoding="utf-8"))
with (data / "occlusion_review_sample_600.csv").open(encoding="utf-8-sig", newline="") as stream:
    full = list(csv.DictReader(stream))
with (data / "occlusion_review_accepted_582.csv").open(encoding="utf-8-sig", newline="") as stream:
    accepted = list(csv.DictReader(stream))
assert results["signature"] == integrity["signature"]
assert not results["trained_or_ran_inference"]
print("Integridad de archivos verificada; análisis del modelo fijo V6.")

Integridad de archivos verificada; análisis del modelo fijo V6.


## Decisiones humanas y soporte

Los 18 casos sin nivel se excluyen por instrucción del usuario. Las notas son opcionales bajo esa misma instrucción; no se rellenan. Los ambiguos permanecen en el resumen descriptivo y no se asignan a baja, media o alta.

In [2]:
excluded = set(acceptance["excluded_annotation_ids"])
assert len(full) == 600 and len(accepted) == 582 and len(excluded) == 18
assert {row["annotation_id"] for row in full if not row["level"]} == excluded
assert accepted == [row for row in full if row["annotation_id"] not in excluded]
assert acceptance["notes_required"] is False and acceptance["human_notes_fabricated"] is False
case_121 = full[120]
assert case_121["annotation_id"] == protocol["selected"][120]["annotation_id"] == "23098"
assert (case_121["level"], case_121["reviewer"], case_121["visual_evidence"]) == ("medium", "Miguel", "")
counts = Counter(row["level"] for row in accepted)
assert counts == {"low": 340, "medium": 69, "high": 48, "ambiguous": 125}
missing_notes = sum(not row["visual_evidence"].strip() for row in accepted)
assert missing_notes == results["review_status"]["accepted_cases_without_notes"] == 398
labels = {"low": "Baja", "medium": "Media", "high": "Alta", "ambiguous": "Ambigua"}
print(f"Aceptados: {len(accepted)}; excluidos: {len(excluded)}; sin nota: {missing_notes}")
for level, label in labels.items():
    print(f"{label}: {counts[level]} ({counts[level] / len(accepted):.1%})")
print("Caso 121: media, Miguel; nota vacía preservada.")

Aceptados: 582; excluidos: 18; sin nota: 398
Baja: 340 (58.4%)
Media: 69 (11.9%)
Alta: 48 (8.2%)
Ambigua: 125 (21.5%)
Caso 121: media, Miguel; nota vacía preservada.


## Métricas e intervalos condicionales

Se conservan todos los fotogramas de `trans_test`, incluidos fondos sin animales, para que sus falsos positivos cuenten. Los objetos sin revisión, excluidos, ambiguos o de otro nivel se ignoran mediante las reglas COCO congeladas. AP no usa los pesos de muestreo. Los intervalos del 95 % remuestrean secuencias completas, con 200 réplicas y semilla 42; mantienen fijo el modelo.

In [3]:
for level in ("low", "medium", "high"):
    group = results["sample_detection"][level]
    metrics, uncertainty = group["metrics"], group["uncertainty"]
    assert group["signature"] == results["signature"]
    assert metrics["eligible_objects"] == counts[level] >= 30
    assert uncertainty["replicates"] == 200 and uncertainty["seed"] == 42
    assert uncertainty["confidence_level"] == 0.95
    print(f"{labels[level]}: {metrics['eligible_objects']} objetos; {uncertainty['clusters']} secuencias")
    for key in ("map50", "map50_95"):
        bounds = uncertainty["intervals"][key]
        assert metrics[key] is not None and bounds["lower"] is not None and bounds["upper"] is not None
        assert bounds["lower"] <= bounds["upper"] and 0 < bounds["defined_replicates"] <= 200
        print(f"  {key}: {metrics[key]:.4f}; IC95% [{bounds['lower']:.4f}, {bounds['upper']:.4f}]; réplicas definidas {bounds['defined_replicates']}/200")

Baja: 340 objetos; 7438 secuencias
  map50: 0.1582; IC95% [0.1457, 0.2117]; réplicas definidas 200/200
  map50_95: 0.0875; IC95% [0.0809, 0.1217]; réplicas definidas 200/200
Media: 69 objetos; 7438 secuencias
  map50: 0.0582; IC95% [0.0389, 0.1035]; réplicas definidas 200/200
  map50_95: 0.0271; IC95% [0.0184, 0.0523]; réplicas definidas 200/200
Alta: 48 objetos; 7438 secuencias
  map50: 0.0563; IC95% [0.0329, 0.1164]; réplicas definidas 200/200
  map50_95: 0.0244; IC95% [0.0160, 0.0567]; réplicas definidas 200/200


## Alcance del resumen descriptivo

Las proporciones observadas se refieren a los 582 casos aceptados. Se conservan los pesos originales para describir su distribución, pero las 18 exclusiones son falta de respuesta humana y no constituyen una nueva muestra aleatoria. No se reutiliza la garantía de planificación de 600 casos ni se afirma representatividad poblacional después de las exclusiones.

In [4]:
descriptive = results["descriptive_occlusion"]
assert descriptive["confidence_intervals_for_population_proportions"] is None
assert descriptive["original_95pct_plus_minus_5pp_planning_guarantee_claimed"] is False
for level, label in labels.items():
    record = descriptive["levels"][level]
    print(f"{label}: observada {record['observed_case_proportion']:.2%}; razón de pesos entre aceptados {record['original_weight_ratio_among_accepted_cases']:.2%}")
print(f"Fracción del peso original excluido: {descriptive['excluded_design_weight_fraction']:.2%}")
print("No se estimaron intervalos poblacionales de proporciones.")

Baja: observada 58.42%; razón de pesos entre aceptados 59.75%
Media: observada 11.86%; razón de pesos entre aceptados 13.02%
Alta: observada 8.25%; razón de pesos entre aceptados 7.55%
Ambigua: observada 21.48%; razón de pesos entre aceptados 19.68%
Fracción del peso original excluido: 2.38%
No se estimaron intervalos poblacionales de proporciones.


## Limitaciones y entrega

Los intervalos de detección son condicionales al modelo y a las secuencias observadas. No miden desacuerdos entre revisores, variación entre entrenamientos ni rendimiento en cámaras futuras. Las 398 notas ausentes y el desacuerdo original del caso 121 quedan documentados; la decisión del usuario permite continuar sin inventar evidencia. El PDF es un anexo de trabajo para revisión editorial, no una aprobación final del informe ni del docente.